### Welcome

This is the first section of the introductory GeoIPS tutorial, which includes running GeoIPS 
using the CLI, and creating your first plugin! 




### Tutorial Scope

This tutorial does not address running GeoIPS in near real-time. GeoIPS plugins 
are intended to be developed and tested on a specific dataset, and setting up the 
real-time processing infrastructure is a separate conversation. 

This tutorial focuses on: 

  - Installing GeoIPS 
  - Running GeoIPS 
  - Product development and testing 


## System requirements

- **CPU:** 1 CPU
- **RAM:** 40GB as the notebook is written, but could use more if modified.
  Reading the entire full-disk ABI image can take up to 100GB.
- **Disk Space:** 10GB storage space.

### Installing Appropriate Test Datasets

For this tutorial, you'll need GeoIPS test datasets that can be used to produce imagery 
or scientific datasets. 

If just attending the beginner tutorial, you'll need:

  - `test_data_abi`

These datasets might take a while to download so go ahead and download the appropriate 
datasets based on the following commands,.

In [ ]:
%%bash

# This is needed for both the beginner and advanced tutorial
geoips config install test_data_abi --outdir $GEOIPS_TESTDATA_DIR

### Introduction to GeoIPS

GeoIPS is a plugin-based system for processing geolocated data: 
  - produce imagery in several formats (most often PNG). 
  - produce output data products in NetCDF4 format. 
  - extended to add other output formats via plugins. 

![1111𝝁 infrared imagery](./images/conus_infrared.png)

![Himawari-9 CLAVR-x Cloud-Top-Height](./images/ahi_cloud_top_height.png)

![ABI CLAVR-x Cloud Top Height](./images/abi_cloud_top_height.png)

GeoIPS is almost entirely composed of plugins:

  - GeoIPS can be extended by developing new plugins in external python packages. 
  - No need to edit the main GeoIPS code to add new functionality. 
  - Most types of functionality in GeoIPS can be extended (and if something can’t be 
  extended, and you think it should, let us know!). 


### Vocabulary

**YAML**
  - "Yet Another Markdown Language" 
  - A human-readable data serialization language that is often used for writing configuration files 

**Plugin**
  - A Python module or YAML file that defines GeoIPS functionality 
  - Stored in an installable Python package that registers its plugin payload with GeoIPS 

**Interface**
  - A class of Python plugins that modify the same type of functionality within GeoIPS 
  (e.g., “the algorithms interface” or “the colormappers interface”) 
  
**Family**
  - A subset of an interface whose plugins accept different sets of arguments/properties 
  - Planning to deprecate this 'artifact' in the near future.


### Commonly used GeoIPS Plugin Types 

**algorithm**
  - Implements a function that modifies data and outputs new data

**colormapper**
  - Defines a method of applying a colormap to imagery

**feature_annotator**
  - Defines how to plot map features (e.g. coastlines, borders, rivers, etc.)

**gridline_annotator**
  - Defines how to plot gridlines and labels

**interpolator**
  - Defines a method of interpolating data to a sector

**output_formatter**
  - Defines a method for plotting imagery or outputting a data file 

**procflow**
  - Executes a series of steps (plugins) in the order specified in a workflow plugin.
  - Operates on a xarray.DataTree object to store the data and metadata from each step
    in a tree-like fashion
  - Transforms data to expected type for a given plugin and back to a DataTree after 
    that plugin has been executed

**workflow**
  - Defines the order of operations in which a procflow will execute

**reader**
  - Defines a data reader 

**sector**
  - Describes an domain for reprojection of data 

## Hands On: Modify a plugin template to create your own installable plugin package  


### Get the Template Repository

To add new plugins to GeoIPS, we use Plugin Packages. You will use a template repository to build your own, installable plugin package containing custom plugins.

The following commands will clone the template plugin repository. Typically, when staring a new plugin package, you would choose the name of your package. We've taken that joy from you and have named your new package "cool_plugins". This name is stored in `$MY_PKG_NAME` for later use. 

`$MY_PKG_DIR` is also provided for convenience and contains the full path to your new package.



In [ ]:
%%bash

cd $GEOIPS_PACKAGES_DIR
source ~/.bashrc

if [[ ! -d "template_basic_plugin" && ! -d "$MY_PKG_NAME" ]]; then
    echo "Cloning template_basic_plugin package"
    git clone --no-tags --single-branch $GEOIPS_REPO_URL/template_basic_plugin.git
else
    echo "Package already exists"
fi

# Rename your package
if [[ ! -d "$MY_PKG_NAME" ]]; then
    echo "Renaming template_basic_plugin to $MY_PKG_NAME"
    mv -v template_basic_plugin/ $MY_PKG_NAME
else
    echo "Package already renamed"
fi

# This will remove references to our upstream repository for safety's sake
cd $MY_PKG_DIR
git remote remove origin 2> /dev/null || true

In [ ]:
%%bash

source ~/.bashrc
echo $MY_PKG_NAME

### Update the Package Name

Now, we can use `ls` to look around in your package directory.


In [ ]:
%%bash

source ~/.bashrc
cd $MY_PKG_DIR
ls --color

This package is set up to be an installable Python package named "my_package". Let's update it to install as "cool_plugins" instead.

Rename the default plugin package directory to "cool_plugins", then call `tree` to see the entire directory structure.

In [ ]:
%%bash

source ~/.bashrc
cd $MY_PKG_DIR

# Rename the package directory
if [[ -d "my_package" ]]; then
    echo "Moving my_package to $MY_PKG_NAME"
    git mv my_package $MY_PKG_NAME
else
    echo "Already moved"
fi

# Show the directory tree two-levels deep
tree -L 2

In [ ]:
%%bash

source ~/.bashrc
cd $MY_PKG_DIR/cool_plugins
tree

![plugins directory structure](./images/plugins_directory_structure.png)

### Update Pertinent Files

1. Update README.md (`vim README.md`)
  - Find/replace all occurrences of @package@  with your package name 
  - Vim Tip :%s/@package@/cool_plugins/g 
  
**Note**: The @ symbols are for ease of searching, take them out when you put your 
package name in!

2. Update pyproject.toml (`vim pyproject.toml`, more on this in soon.)
  - Find/replace all occurrences of my_package  with your package name 

3. Add and commit your changes.

#### Note

To save time at this point in the tutorial, we will copy these updated files
into place.

In [ ]:
"""Overwrite cool_plugins' pyproject.toml and README.md with correct contents."""

import os

with open("./updated_files/pyproject.toml", "r") as rf:
    toml_lines = rf.readlines()

with open(f"{os.environ['GEOIPS_PACKAGES_DIR']}/cool_plugins/pyproject.toml", "w") as wf:
    wf.writelines(toml_lines)

with open("./updated_files/README.md", "r") as rf:
    md_lines = rf.readlines()

with open(f"{os.environ['GEOIPS_PACKAGES_DIR']}/cool_plugins/README.md", "w") as wf:
    wf.writelines(md_lines)

In [ ]:
%%bash

source ~/.bashrc
cd $MY_PKG_DIR
git add README.md pyproject.toml
git commit -m "Updated name of template plugin package to mine" || true

#### Install your package 
Now that your package is updated, you can install it! 

We'll use `pip install -e $MY_PKG_DIR` where `-e` means "editable". This installs the package in "editable" mode so we can edit the package after it is installed and changes will be reflected in the installed package.

*For those who are interested, this acts similarly to a symlink in Linux, but has some more complexity behind it.*

In [ ]:
%%bash

source ~/.bashrc
pip install -e $MY_PKG_DIR

### See what you just installed 

Use the GeoIPS CLI to list the installed packages. Yours should be there!

In [ ]:
%%bash

geoips list packages

### A bit about pyproject.toml

Installing Python packages requires metadata that describes the package and how to 
install it. 

`pyproject.toml` defines this information for pip, including: 
  - Package name, version, description, license, etc. 
  - Which files should be contained in the package when installed 
  - How to build the package 

We make GeoIPS aware of our package using the `geoips.plugin_packages` namespace.
This allows GeoIPS to find all plugins within packages registered to this namespace.

GeoIPS automatically identifies all plugins defined within a plugin package via a plugin
registry. You can manually create these files via `geoips config create-registries`, 
however, GeoIPS will automatically create these files if a requested plugin cannot be
found. This usually occurs the first time GeoIPS is initialized.

**NOTE** for plugin registries to write successfully: 

1. All installed  plugin names within a given interface must be unique 

2. All installed plugins must be formatted and defined correctly 

We will make use of this more later! Modify plugin template solutions on 
[NRLMMD github.com](https://github.com/NRLMMD-GEOIPS/template_basic_plugin/tree/workshop-2023-solutions).

```
[tool.poetry.plugins."geoips.plugin_packages"]
"cool_plugins" = "cool_plugins" 
``` 

## GeoIPS Command Line Interface (CLI) Tutorial

The GeoIPS CLI can provide you with a lot of information about the installed plugin packages and their plugins. Please follow this [Jupyter Notebook](./CLI_Tutorial.ipynb) for instructions on how to 
make use of the GeoIPS CLI.

## Hands on: Create a Workflow Plugin (workflows  YAML-based interface) 

Solutions on 
[NRLMMD github.com](https://github.com/NRLMMD-GEOIPS/template_basic_plugin/tree/workshop-2026-solutions).

### First Workflow Plugin: Severe Storms RGB imagery from G16 ABI data

Let's start by copying an existing workflow plugin to a new file. We'll modify it to create a new plugin.

In [ ]:
%%bash

source ~/.bashrc
mkdir -p $MY_PKG_DIR/$MY_PKG_NAME/plugins/yaml/workflows
cp -v $MY_PKG_DIR/$MY_PKG_NAME/plugins/yaml/products/amsr2_using_product_defaults.yaml  $MY_PKG_DIR/$MY_PKG_NAME/plugins/yaml/workflows/My-ABI-Severe-Storms.yaml

### GeoIPS Yaml-based plugin properties

All YAML plugins will begin with these same four properties as shown below.

![Workflow top level keys](./images/workflow_top_level.png)

### Edit my_clavrx_products.yaml properties

Update `My-ABI-Severe-Storms.yaml` to match the following four lines *(Feel free to remove all lines preceded by `# @`)*:

```yaml
interface: workflows
family: order_based
name: My-ABI-Severe-Storms
docstring: |
  ABI Severe Storms RGB workflow.
```

**Click to edit in a new tab:
[My-ABI-Severe-Storms.yaml](../../cool_plugins/cool_plugins/plugins/yaml/workflows/My-ABI-Severe-Storms.yaml)
Close and save when done!**

### First Workflow Plugin: Severe Storms RGB imagery from G16 ABI data

Next, let's add our first new workflow! We'll start by adding a series of steps which will produce GOES ABI Severe Storms RGB imagery. To do so, update the existing workflow as shown below.

Edit the file (**click me:** [My-ABI-Severe-Storms.yaml](../../cool_plugins/cool_plugins/plugins/yaml/products/My-ABI-Severe-Storms.yaml)) again to update the workflow specification to match the following lines:

```yaml
spec:
  steps:
    retrieve_area_definition: # Step ID. Should denote what the step performs.
      kind: sector            # The type of plugin being applied. Singular.
      name: goes_east         # The name of the plugin of type 'kind'.
    read_abi_data:
      kind: reader
      name: abi_netcdf
      arguments:
        chans: &variable-list ['B08BT', 'B10BT', 'B07BT', 'B13BT', 'B05Ref', 'B02Ref']
        resampled_read: True
    apply_nn_interpolator:
      kind: interpolator
      name: interp_nearest
      depends_on: [read_abi_data, retrieve_area_definition]
      arguments:
        varlist: *variable-list # Some plugins require arguments. 'varlist' should always be provided for an interpolator plugin.
    apply_severe_storms_algorithm:
      kind: algorithm
      name: severe_storms
      arguments:
        # The severe storms algorithm (to be created) requires these argument fields.
        red:
          range: [-35.0, 5.0]
          units: Kelvin
          gamma: 1
        green:
          range: [5.0, 60.0]
          units: Kelvin
          gamma: 0.5
        blue:
          range: [-75.0, 25.0]
          units: percentage
          gamma: 1
    retrieve_colormap:
      kind: colormapper
      name: cmap_rgb
    retrieve_gridlines:
      kind: gridline_annotator
      name: default
    retrieve_features:
      kind: feature_annotator
      name: default
    apply_imagery_annotated:
      kind: output_formatter
      name: imagery_annotated
      depends_on:
        # The output formatter takes a bunch of different plugin types to produce a final output
        - apply_severe_storms_algorithm
        - retrieve_colormap
        - retrieve_gridlines
        - retrieve_features
        - retrieve_area_definition
      arguments:
        product_name: My-ABI-Severe-Storms
        product_name_title: G16 Severe Storms RGB
        output_fnames: [!ENV $GEOIPS_OUTDIRS/abi_severe_storms.png]
```

### Update the Plugin Registry

Let's use the CLI to get more information about the plugin we just created.

While this won't be needed in the future (there is a bug...) we need to first rebuild the plugin registry to allow GeoIPS to locate your new plugin.

In [ ]:
!geoips config create-registry

### List the plugins

To ensure that your new plugin was installed and registered, you can call `geoips list workflows` or `geoips ls workflows`. With just that, you will list all workflows from all packages, though. To see only the plugins from the `cool_plugins` package, we add the `-p cool_plugins` option.

You should see your new plugin in the output below!

In [ ]:
!geoips ls workflows -p cool_plugins

### Describing plugins

To get more information about a particular plugin (or interface), call `geoips describe`. This is the generic format to follow via the CLI:

`geoips describe <interface_name> <source_name>.<plugin_name>`

In [ ]:
!geoips describe workflow ABI-Infrared

## Adding a New Algorithm Plugin

You might not have noticed this before, but one of the plugin steps we referenced in our new workflow doesn't actually exist. This is the ``apply_severe_storms_algorithm`` step which references a ``severe_storms`` algorithm. If you run ``geoips list algorithms`` you'll see that no such algorithm exists. We are going to create that plugin right now.

In [ ]:
%%bash

source ~/.bashrc
cd $MY_PKG_DIR/$MY_PKG_NAME/plugins
mkdir -p classes/algorithms
cp -v modules/algorithms/pmw_89test.py classes/algorithms/severe_storms.py

### The same three top-level attributes
Just like the Yaml-based plugins, this class-based plugin has three top-level attributes that are common to all GeoIPS plugins: interface, family, and name. Here:
- `interface` is "algorithms" to indicate that this plugin belongs to the Algorithms interface.
- `family` is "xarray_to_numpy", a common family for algorithms indicating that it accepts an Xarray DataSet as input and returns numpy ndarray as output.
- `name` is the name of the algorithm (which we will update).

![Updating top level portions of new algorithm](./images/alg_screenshot.png)

### Let's update our `severe_storms` algorithm with the following:

```python
"""Severe Storms WMO RGB recipe."""

from geoips.interfaces.class_based.algorithms import BaseAlgorithmPlugin

import logging

LOG = logging.getLogger(__name__)


class SevereStormsAlgorithmPlugin(BaseAlgorithmPlugin):
    """Severe Storms WMO RGB recipe."""

    interface = "algorithms"
    family = "xarray_to_numpy"
    name = "severe_storms"
```

**Edit me: [severe_storms.py](../../cool_plugins/cool_plugins/plugins/classes/algorithms/severe_storms.py) (Close and save the file when done)**

### Updating your Algorithm

All module and class-based plugins, including Algorithms, must include a `call()` function. This function is what is called when the plugin is executed. The `call()` function's signature is determined by the algorithm's family.

We will largely trim what arguments are present in this plugin's call signature. All we need are the ``red``, ``green``, and ``blue`` argument dictionaries we defined in our workflow.

```python
    def call(self, xobj, red, green, blue):
        """Apply WMO's Severe Storms RGB recipe.

        Parameters
        ----------
        xobj : xarray.Dataset
            The dataset containing variables needed for the severe storms rgb recipe.
        red : dict
            A dictionary containing {'range', 'units', 'gamma'} key value pairs which
            define the parameters to the red gun of a RGB recipe.
        green : dict
            A dictionary containing {'range', 'units', 'gamma'} key value pairs which
            define the parameters to the green gun of a RGB recipe.
        blue : dict
            A dictionary containing {'range', 'units', 'gamma'} key value pairs which
            define the parameters to the blue gun of a RGB recipe.

        Returns
        -------
        numpy.ndarray
            numpy.ndarray or numpy.MaskedArray of qualitative RGBA image output
        """
```

**Edit me: [severe_storms.py](../../cool_plugins/cool_plugins/plugins/classes/algorithms/severe_storms.py) (Close and save the file when done)**

### Update the call function's functionality

Inside the `call()` is where the actual data manipulation occurs. To update the algorithm to produce Severe Storms RGB from input data, replace the contents of the `call()` function with the following. If you examine the code below, you will see that it:
- Extracts the relevant variables from the xarray object.
- Applies the appropriate equation for each RGB gun
- Converts input data to expected units
- Crops data to its appropriate min / max range
- Applies gamma corrections to each RGB gun
- Returns the data as an ``[[r], [g], [b], [a]]`` np.ndarray

The returned will now be a Severe Storms RGBA tuple.

Add the following contents to your plugin's ``call()`` function.

```python
        rparams = red
        gparams = green
        bparams = blue

        red = xobj["B08BT"].to_masked_array() - xobj["B10BT"].to_masked_array()
        grn = xobj["B07BT"].to_masked_array() - xobj["B13BT"].to_masked_array()
        blu = xobj["B05Ref"].to_masked_array() - xobj["B02Ref"].to_masked_array()

        # Ensure red and green guns are in Kelvin units
        from geoips.data_manipulations.conversions import unit_conversion

        red = unit_conversion(red, input_units="Kelvin", output_units=rparams["units"])
        grn = unit_conversion(grn, input_units="Kelvin", output_units=gparams["units"])
        # No unit conversion needed to be applied to blue gun

        from geoips.data_manipulations.corrections import apply_data_range, apply_gamma

        data_range = rparams["range"]
        gamma = rparams["gamma"]
        red = apply_data_range(
            red,
            min_val=data_range[0],
            max_val=data_range[1],
            min_outbounds="crop",
            max_outbounds="crop",
            norm=True,
            inverse=False,
        )
        red = apply_gamma(red, gamma)

        data_range = gparams["range"]
        gamma = gparams["gamma"]
        grn = apply_data_range(
            grn,
            min_val=data_range[0],
            max_val=data_range[1],
            min_outbounds="crop",
            max_outbounds="crop",
            norm=True,
            inverse=False,
        )
        grn = apply_gamma(grn, gamma)

        data_range = bparams["range"]
        gamma = bparams["gamma"]
        blu = apply_data_range(
            blu,
            min_val=data_range[0],
            max_val=data_range[1],
            min_outbounds="crop",
            max_outbounds="crop",
            norm=True,
            inverse=False,
        )
        blu = apply_gamma(blu, gamma)

        from geoips.image_utils.mpl_utils import (
            alpha_from_masked_arrays,
            rgba_from_arrays,
        )

        alp = alpha_from_masked_arrays([red, grn, blu])
        rgba = rgba_from_arrays(red, grn, blu, alp)

        return rgba


# Tells pluginify (plugin registry package) what object is the actual plugin we want to
# add to the registry
PLUGIN_CLASS = SevereStormsAlgorithmPlugin
```

**Edit me: [severe_storms.py](../../cool_plugins/cool_plugins/plugins/classes/algorithms/severe_storms.py) (Close and save the file when done)**

Once complete, let's describe the plugin you just created to make sure it registered appropriately.

In [ ]:
%%bash

geoips config create-registries
# In case you forgot the name of your plugin, you can run:
geoips ls algs -p cool_plugins
# Now let's describe that plugin
geoips describe algorithm severe_storms

### Use your new workflow!

Now that we've created the missing algorithm plugin, the workflow we created is able
to be ran. We'll do this via the ``geoips run`` command.

- GeoIPS is called via a command line interface (CLI), as we've shown previously.
- The main command that you will use is ``geoips run``, which will run your 
  data through the order based procflow (OBP) using the specified plugins.
- It's easiest to do this via a script, and scripts are stored in your plugin package's 
  `tests/` directory because they can be used later to regression test your package 
- Since we're running this in a notebook, we can easily run these commands with a code cell!

In [ ]:
%%bash


source ~/.bashrc
geoips run order_based My-ABI-Severe-Storms $GEOIPS_TESTDATA_DIR/test_data_abi/data/goes16_20200918_1950/*

### Viewing the log output

This will write some log output.  If your script succeeded it will end with 
`INTERACTIVE: Return Value 0`.

To view your output, look for a line that says `SINGLESOURCESUCCESS` and open the file shown there (or run the cell below).

If successful, the output image should look like this:

![CLAVR-x CONUS My-Cloud-Top-Height](./images/abi_severe_storms.png)

In [ ]:
from IPython.display import Image

Image(f"{os.environ['GEOIPS_OUTDIRS']}/abi_severe_storms.png")